Learn to predict the next character in the works of William Shakespeare, and use the acquired model to generate samples of Shakespeare-like text.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical
import numpy as np
#try to use a GPU, if it's available, for faster training. may not be.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
#make sure you have the Shakespeare.txt file somewhere accessible. You can upload it using the functionality on the left <----

#alternative
# from google.colab import drive
# drive.mount('/content/drive/MyDrive/Shakespeare.txt')

data_path = ('/content/uzbek_corpus.txt')

#if you have trouble with the data path. use the command "pwd" (present working directory) to see where you are, and make sure the path makes sense, given that location.
#you can find the file on the left and click "copy path"

In [ ]:
#we use a version of an RNN called a "Long-Short Term Memory".
#Like a simple RNN, an LSTM processes sequences of data, and continually uses the previous hidden state activitations to causally influence the activities at each subsequent "timestep" in the sequence.

#but, unlike a simple RNN, it additionally has "gating" parameters that allow the information-flow to be dynamically modified. In most cases, it performs better than a simple RNN.

class LSTM(nn.Module):
    def __init__(self, input_size, output_size, hidden_size, num_layers):
        super(LSTM, self).__init__()
        self.embedding = nn.Embedding(input_size, input_size)
        self.rnn = nn.LSTM(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers)
        self.decoder = nn.Linear(hidden_size, output_size)

    def forward(self, input_seq, hidden_state):
        embedding = self.embedding(input_seq)
        output, hidden_state = self.rnn(embedding, hidden_state)
        output = self.decoder(output)
        return output, (hidden_state[0].detach(), hidden_state[1].detach())


In [ ]:
#features of the model that you could modify, if you wished.

hidden_size = 512   # size of hidden state
seq_len = 100       # length of LSTM sequence
num_layers = 3      # num of layers in LSTM
lr = 0.002          # learning rate; how big of a step to take in gradient descent
epochs = 500        # max number of epochs
op_seq_len = 100    # total num of characters in output test sequence
load_chk = False   # load weights from save_path directory to continue training

#where the model should be saved (or already was saved, if you are re-running)
# from google.colab import drive
# drive.mount('/content/drive/MyDrive/Shakespeare.txt')
#save_path = "/content/drive/MyDrive/CharRNN_Shakespeare"

In [ ]:
# load the text file
data = open(data_path, 'r').read()
chars = sorted(list(set(data)))
data_size, vocab_size = len(data), len(chars)
print("----------------------------------------")
print("Data has {} characters, {} unique".format(data_size, vocab_size))
print("----------------------------------------")


----------------------------------------
Data has 3982533 characters, 122 unique
----------------------------------------


In [ ]:
# char to index and index to char maps
char_to_ix = { ch:i for i,ch in enumerate(chars) }
ix_to_char = { i:ch for i,ch in enumerate(chars) }

# convert data from characters to indices
data = list(data)
for i, ch in enumerate(data):
    data[i] = char_to_ix[ch]

# data tensor on device
data = torch.tensor(data).to(device)
data = torch.unsqueeze(data, dim=1)

#################################

In [ ]:
# model
rnn = LSTM(vocab_size, vocab_size, hidden_size, num_layers).to(device)

# if you already ran the model and saved the progress, you can re-load it here.
if load_chk:
    rnn.load_state_dict(torch.load(save_path))
    print("Model loaded successfully !!")
    print("----------------------------------------")

# loss function and optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(rnn.parameters(), lr=lr)

In [ ]:
# main training loop for recurrent neural network
for i_epoch in range(1, epochs+1):
    sample_n = 0
    # random starting point
    data_ptr = np.random.randint(100)
    n = 0
    running_loss = 0
    hidden_state = None

    while True:
      #given one character as input,
        input_seq = data[data_ptr : data_ptr+seq_len]
      #predict the next character in the sequence
        target_seq = data[data_ptr+1 : data_ptr+seq_len+1]

      # forward pass through neural network to generate each prediction
        output, hidden_state = rnn(input_seq, hidden_state)

    ##### the loss is the difference between the predicted next character and the true next character.###
        loss = loss_fn(torch.squeeze(output), torch.squeeze(target_seq))
        running_loss += loss.item()

    #compute gradients and backprop error
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    #update the data pointer (Where in sequence are we?)
        data_ptr += seq_len
        n +=1

    #if at end of data : break
        if data_ptr + seq_len + 1 > data_size:
            break

        if n % 100 ==0:
          sample_n+=1

    ###### generate some text, occasionally
          data_ptr_test = 0
          hidden_state = None

      #random character from data to begin generating a sample
          rand_index = np.random.randint(data_size-1)
          input_seq = data[rand_index : rand_index+1]

          print("----------------------------------------")
          print("Sample Number: {0} \t Loss: {1:.8f}".format(sample_n, running_loss/n))
          while True:
          # forward pass
              output, hidden_state = rnn(input_seq, hidden_state)

          #sample a character
              output = F.softmax(torch.squeeze(output), dim=0)
              dist = Categorical(output)
              index = dist.sample()

          # print the sampled character
              print(ix_to_char[index.item()], end='')

          # next input is current output
              input_seq[0][0] = index.item()
              data_ptr_test += 1

              if data_ptr_test > op_seq_len:
                  break
          print("----------------------------------------")

# print loss and save weights after every epoch
    print("Epoch: {0} \t Loss: {1:.8f}".format(i_epoch, running_loss/n))
    torch.save(rnn.state_dict(), save_path)


----------------------------------------
Sample Number: 1 	 Loss: 3.29701073
vmrryistusknk? nnhi  aoeit kh  a t
NhirnOpiiiaqhkn t.stnoa  a zixqoq
 tnajqddha qkhtdns‘ rJ rgtiliuai----------------------------------------
----------------------------------------
Sample Number: 2 	 Loss: 3.19124595
vik tu
lo‘eganunci hti bire`enig e.sheda sefi.k’il kon‘s oyBib ‘yiq eatg kos kanud yodossgu.Qopocor b----------------------------------------
----------------------------------------
Sample Number: 3 	 Loss: 2.96970249
himi echnga i.
Bada nani muazman.n alemi sech?
Kavsandn shiris chqa us virtorsaris Mo‘lda.
Titki Yell----------------------------------------
----------------------------------------
Sample Number: 4 	 Loss: 2.78980447
?Unini tilchoy ha sardi.
So supsaz hoqgiyiytisiyush,ia ushiksay oliz?
Yim kexra uymi.
Veringili fovla----------------------------------------
----------------------------------------
Sample Number: 5 	 Loss: 2.65947202
-hetan kib yo‘rda.
Umadan bu mo‘satir, nashmaq.